# Module 7: Change Point Detection in Commodity Markets

## Learning Objectives
By completing this notebook, you will:
1. Detect structural breaks in commodity price time series
2. Implement Bayesian change point models in PyMC
3. Distinguish between gradual regime transitions and sudden breaks
4. Apply change point detection to policy shifts and market disruptions
5. Forecast with regime-aware models

## Prerequisites
- Hidden Markov Models (previous notebook)
- Bayesian inference
- Commodity market structure

## Estimated Time: 75 minutes

---

In [ ]:
learning_objectives(["Detect structural breaks in commodity price time series", "Implement Bayesian change point models in PyMC", "Distinguish between gradual regime transitions and sudden breaks", "Apply change point detection to policy shifts and market disruptions", "Forecast with regime-aware models", "Hidden Markov Models (previous notebook)"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az
from scipy import stats

np.random.seed(42)
az.style.use('arviz-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"PyMC version: {pm.__version__}")

## Conceptual Introduction

### What is a Change Point?

**Definition:** A time $\tau$ when statistical properties of a time series change abruptly.

**Examples in Commodity Markets:**
1. **Policy shifts** - Saudi Arabia ends oil price band (2014)
2. **Technology adoption** - Shale oil boom (2008-2014)
3. **Regulatory changes** - Renewable Fuel Standard mandates (2007)
4. **Market structure** - Financialization of commodities (2004-2008)
5. **Supply shocks** - Russian invasion of Ukraine (2022)

### Change Point vs. HMM

| Feature | Change Point | HMM |
|---------|-------------|-----|
| Transitions | Permanent | Reversible |
| Number of breaks | Few (1-3) | Many (continuous switching) |
| Use case | Structural breaks | Business cycles |
| Example | Policy change | Contango/backwardation |

### Bayesian Change Point Model

**Single Change Point:**

$$y_t \sim \begin{cases}
\mathcal{N}(\mu_1, \sigma_1^2) & t < \tau \\
\mathcal{N}(\mu_2, \sigma_2^2) & t \geq \tau
\end{cases}$$

**Prior on change point:**
$$\tau \sim \text{DiscreteUniform}(1, T)$$

**Why Bayesian?**
- Uncertainty over $\tau$ ("when did the break occur?")
- Uncertainty over regime parameters $(\mu_1, \sigma_1, \mu_2, \sigma_2)$
- Can incorporate prior knowledge (e.g., "break likely after 2014")

## Part 1: Single Change Point Model

In [ ]:
# Purpose: Generate synthetic commodity data with known change point
# Key Concept: Abrupt shift in mean and volatility

n = 200
true_tau = 120  # Change point at t=120

# Regime 1 (before break): Low mean, low volatility
mu_1 = 50.0
sigma_1 = 2.0

# Regime 2 (after break): High mean, high volatility
mu_2 = 65.0
sigma_2 = 5.0

# Generate data
y = np.zeros(n)
y[:true_tau] = np.random.normal(mu_1, sigma_1, true_tau)
y[true_tau:] = np.random.normal(mu_2, sigma_2, n - true_tau)

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Plot 1: Time series with true change point
ax = axes[0]
ax.plot(y, 'o-', markersize=4, linewidth=1, alpha=0.7)
ax.axvline(true_tau, color='red', linestyle='--', linewidth=3, label=f'True Change Point (t={true_tau})')
ax.axhline(mu_1, color='blue', linestyle=':', linewidth=2, alpha=0.7, label='Regime 1 Mean')
ax.axhline(mu_2, color='green', linestyle=':', linewidth=2, alpha=0.7, label='Regime 2 Mean')
ax.set_xlabel('Time', fontsize=12)
ax.set_ylabel('Price', fontsize=12)
ax.set_title('Commodity Prices with Structural Break', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Rolling statistics (shows break visually)
ax = axes[1]
window = 20
rolling_mean = pd.Series(y).rolling(window).mean()
rolling_std = pd.Series(y).rolling(window).std()

ax.plot(rolling_mean, linewidth=2, label='Rolling Mean (20-period)', color='blue')
ax_twin = ax.twinx()
ax_twin.plot(rolling_std, linewidth=2, label='Rolling Std (20-period)', color='orange')
ax.axvline(true_tau, color='red', linestyle='--', linewidth=3, alpha=0.7)

ax.set_xlabel('Time', fontsize=12)
ax.set_ylabel('Rolling Mean', fontsize=12, color='blue')
ax_twin.set_ylabel('Rolling Std', fontsize=12, color='orange')
ax.set_title('Rolling Statistics Reveal Structural Break', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left')
ax_twin.legend(loc='upper right')

plt.tight_layout()
plt.show()

print(f"True parameters:")
print(f"Regime 1 (t < {true_tau}): μ={mu_1}, σ={sigma_1}")
print(f"Regime 2 (t ≥ {true_tau}): μ={mu_2}, σ={sigma_2}")
print(f"\nChallenge: Infer τ, μ_1, σ_1, μ_2, σ_2 from data alone")

In [ ]:
# Purpose: Build Bayesian change point model
# Key Concept: Discrete latent variable for change point location

with pm.Model() as changepoint_model:
    # Prior on change point location
    # Constrain to avoid edge effects (not first/last 10% of data)
    tau = pm.DiscreteUniform('tau', lower=int(0.1*n), upper=int(0.9*n))
    
    # Regime 1 parameters
    mu_1_param = pm.Normal('mu_1', mu=50, sigma=10)
    sigma_1_param = pm.HalfNormal('sigma_1', sigma=5)
    
    # Regime 2 parameters
    mu_2_param = pm.Normal('mu_2', mu=60, sigma=10)
    sigma_2_param = pm.HalfNormal('sigma_2', sigma=5)
    
    # Regime indicator: 0 before tau, 1 after
    regime = pm.math.switch(pm.math.arange(n) < tau, 0, 1)
    
    # Regime-dependent parameters
    mu = pm.math.switch(regime, mu_2_param, mu_1_param)
    sigma = pm.math.switch(regime, sigma_2_param, sigma_1_param)
    
    # Likelihood
    pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y)
    
    # Sample
    trace_cp = pm.sample(
        2000,
        tune=2000,
        target_accept=0.95,
        return_inferencedata=True,
        random_seed=42
    )

print("\nPosterior summary:")
print(az.summary(
    trace_cp,
    var_names=['tau', 'mu_1', 'sigma_1', 'mu_2', 'sigma_2'],
    round_to=2
))

In [ ]:
# Purpose: Visualize change point posterior
# Key Concept: Uncertainty over change point location

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Posterior distribution of tau
ax = axes[0, 0]
tau_samples = trace_cp.posterior['tau'].values.flatten()
ax.hist(tau_samples, bins=50, density=True, alpha=0.7, edgecolor='black')
ax.axvline(true_tau, color='red', linestyle='--', linewidth=3, label=f'True τ={true_tau}')
ax.axvline(tau_samples.mean(), color='blue', linestyle='-', linewidth=2, 
          label=f'Posterior Mean={tau_samples.mean():.1f}')
ax.set_xlabel('Change Point Location (τ)', fontsize=12)
ax.set_ylabel('Posterior Density', fontsize=12)
ax.set_title('When Did the Break Occur?', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Regime parameter posteriors
ax = axes[0, 1]
az.plot_forest(
    trace_cp,
    var_names=['mu_1', 'mu_2', 'sigma_1', 'sigma_2'],
    combined=True,
    ax=ax
)
true_vals = [mu_1, mu_2, sigma_1, sigma_2]
for i, true_val in enumerate(true_vals):
    ax.axvline(true_val, color='red', linestyle='--', alpha=0.7)
ax.set_title('Regime Parameters (Red = True)', fontsize=12, fontweight='bold')

# Plot 3: Data with inferred regimes
ax = axes[1, 0]
tau_mean = int(tau_samples.mean())
mu_1_est = trace_cp.posterior['mu_1'].mean().item()
mu_2_est = trace_cp.posterior['mu_2'].mean().item()

ax.plot(y, 'o-', markersize=4, linewidth=1, alpha=0.6, label='Data')
ax.axvline(tau_mean, color='blue', linestyle='-', linewidth=3, label=f'Inferred τ={tau_mean}')
ax.axvline(true_tau, color='red', linestyle='--', linewidth=2, label=f'True τ={true_tau}', alpha=0.7)
ax.axhline(mu_1_est, color='blue', linestyle=':', linewidth=2, alpha=0.7)
ax.axhline(mu_2_est, color='green', linestyle=':', linewidth=2, alpha=0.7)
ax.set_xlabel('Time', fontsize=12)
ax.set_ylabel('Price', fontsize=12)
ax.set_title('Data with Inferred Change Point', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: Regime probability over time
ax = axes[1, 1]
# For each time point, compute P(regime 2 | data)
regime_prob = np.zeros(n)
for t in range(n):
    regime_prob[t] = (tau_samples <= t).mean()

ax.plot(regime_prob, linewidth=3, color='purple')
ax.axvline(true_tau, color='red', linestyle='--', linewidth=2, label='True Break', alpha=0.7)
ax.set_xlabel('Time', fontsize=12)
ax.set_ylabel('P(Regime 2 | Data)', fontsize=12)
ax.set_title('Probability of Being in High-Price Regime', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

print(f"\nModel successfully identified change point:")
print(f"True: τ={true_tau}")
print(f"Estimated: τ={tau_mean} (95% CI: [{np.percentile(tau_samples, 2.5):.0f}, {np.percentile(tau_samples, 97.5):.0f}])")

## Exercise 7.1: Multiple Change Points

**Task:** Extend the model to detect TWO change points. Generate data with:
- Regime 1 (t < 80): μ=50, σ=2
- Regime 2 (80 ≤ t < 140): μ=70, σ=4
- Regime 3 (t ≥ 140): μ=55, σ=3

Build a PyMC model with:
- Two change points: τ_1 < τ_2
- Three sets of regime parameters

**Expected Output:**
- Model identifies both break points
- Regime parameters correctly estimated
- Uncertainty captured in credible intervals

**Hints:**
<details>
<summary>Hint 1</summary>
Use ordered discrete uniforms: tau_1 ~ DiscreteUniform(0, 100), tau_2 ~ DiscreteUniform(tau_1, n)
</details>

<details>
<summary>Hint 2</summary>
Regime indicator: pm.math.switch(t < tau_1, 0, pm.math.switch(t < tau_2, 1, 2))
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# 1. Generate data with 2 change points
# 2. Build PyMC model with ordered change points
# 3. Fit and visualize

n_multi = 200
true_tau_1 = 80
true_tau_2 = 140

# YOUR CODE: Generate data

with pm.Model() as multi_changepoint_model:
    # YOUR CODE: Define model
    pass

your_answer = None  # Replace with trace

In [ ]:
# AUTO-GRADED TESTS
def test_exercise_7_1():
    assert your_answer is not None, "Implement your solution!"
    assert isinstance(your_answer, az.InferenceData), "Return InferenceData"
    
    # Check for two change point parameters
    assert 'tau_1' in your_answer.posterior or 'tau1' in your_answer.posterior, \
        "Need first change point parameter"
    assert 'tau_2' in your_answer.posterior or 'tau2' in your_answer.posterior, \
        "Need second change point parameter"
    
    print("Exercise 7.1 passed!")
    print("Model successfully detected multiple structural breaks")

test_exercise_7_1()

## Part 2: Real-World Application - Oil Price Shocks

Let's apply change point detection to a realistic scenario: detecting the 2014 oil price crash.

In [ ]:
# Purpose: Simulate oil prices with 2014 structural break
# Key Concept: Saudi Arabia abandons price band, prices collapse

# 2012-2016 (monthly data)
n_oil = 60
dates_oil = pd.date_range('2012-01-01', periods=n_oil, freq='MS')

# Before break (2012-2014 mid): High stable prices
# After break (2014 mid - 2016): Low volatile prices
break_date = '2014-07-01'
break_idx = dates_oil.get_loc(break_date)

oil_before = np.random.normal(100, 5, break_idx)
oil_after = np.random.normal(55, 10, n_oil - break_idx)
oil_prices = np.concatenate([oil_before, oil_after])

# Visualize
plt.figure(figsize=(14, 6))
plt.plot(dates_oil, oil_prices, 'o-', linewidth=2, markersize=6)
plt.axvline(pd.Timestamp(break_date), color='red', linestyle='--', 
           linewidth=3, label='True Break (July 2014)', alpha=0.8)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Brent Crude ($/barrel)', fontsize=12)
plt.title('Crude Oil Prices: 2014 Structural Break', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Break occurred at: {break_date} (index {break_idx})")
print(f"Before: ${oil_before.mean():.1f} ± ${oil_before.std():.1f}")
print(f"After: ${oil_after.mean():.1f} ± ${oil_after.std():.1f}")

In [ ]:
# Purpose: Fit change point model to oil data

with pm.Model() as oil_changepoint:
    tau_oil = pm.DiscreteUniform('tau', lower=5, upper=n_oil-5)
    
    mu_before = pm.Normal('mu_before', mu=90, sigma=20)
    sigma_before = pm.HalfNormal('sigma_before', sigma=10)
    
    mu_after = pm.Normal('mu_after', mu=60, sigma=20)
    sigma_after = pm.HalfNormal('sigma_after', sigma=10)
    
    regime_oil = pm.math.switch(pm.math.arange(n_oil) < tau_oil, 0, 1)
    mu_oil = pm.math.switch(regime_oil, mu_after, mu_before)
    sigma_oil = pm.math.switch(regime_oil, sigma_after, sigma_before)
    
    pm.Normal('oil_obs', mu=mu_oil, sigma=sigma_oil, observed=oil_prices)
    
    trace_oil = pm.sample(
        2000,
        tune=2000,
        target_accept=0.95,
        return_inferencedata=True,
        random_seed=42
    )

# Extract results
tau_oil_samples = trace_oil.posterior['tau'].values.flatten()
tau_oil_mean = int(tau_oil_samples.mean())
detected_date = dates_oil[tau_oil_mean]

print(f"\nDetected break:")
print(f"Index: {tau_oil_mean} (true: {break_idx})")
print(f"Date: {detected_date.strftime('%Y-%m')} (true: 2014-07)")
print(f"\nModel successfully identified the 2014 oil price collapse")

## Summary

### Key Takeaways

1. **Change points detect structural breaks** - Permanent shifts in time series properties

2. **Bayesian approach quantifies uncertainty** - Both in timing (τ) and regime parameters

3. **Different from regime switching** - Changes are permanent, not cyclic

4. **Multiple change points** - Can detect sequences of structural breaks

5. **Real-world applications** - Policy changes, market disruptions, technology adoption

### Common Use Cases

| Market | Change Point Event |
|--------|-------------------|
| Crude Oil | 2014 OPEC policy shift |
| Natural Gas | Shale gas revolution (2008) |
| Corn | Renewable Fuel Standard (2007) |
| Electricity | Deregulation (varies by region) |

### What's Next

**Next Notebook: Markov Switching Models** - Combine regime switching with state space models

**Module 8: Fundamentals Integration** - Incorporate fundamental drivers in regime-dependent forecasts

---

*"Markets don't just fluctuate - they transform. Change points mark the transformations."*